# APEX Configuration Toolkit

# Functions

## Water balance accounting

In [ ]:
import glob
import os
import pandas as pd
import numpy as np

def generate_wb_table(source_fp):
    '''
    Create a dataframe with wb components from daily summary output files (.SAD and .DHY)
    source_fp (str): path to the directory containing the .SAD and .DHY files
    returns: dataframe with wb components
    '''

    # locate the .SAD file in the source directory
    sad_file = glob.glob(os.path.join(source_fp, '*.SAD'))
    dhy_file = glob.glob(os.path.join(source_fp, '*.DHY'))

    # read in files as df
    sad_df = pd.read_csv(sad_file[0], sep='\s+', skiprows=9, encoding='latin1')
    dhy_df = pd.read_csv(dhy_file[0], sep='\s+', skiprows=8, encoding='latin1')

    # create new df with only the variables of interest
    wb_df = sad_df[['Y', 'M', 'D', 'PRCP', 'WYLD', 'DPRK', 'ET', 'SNOF', 'SW', 'SWLT','PET']]

    # add GWST from the .DHY file
    wb_df['GWST'] = dhy_df['GWSTmm']
    wb_df['RFV'] = dhy_df['RFVmm']


    # calculate storage changes
    wb_df['ΔSNOF'] = wb_df['SNOF'].diff()
    wb_df['ΔSW'] = wb_df['SW'].diff()
    wb_df['ΔSWLT'] = wb_df['SWLT'].diff()
    wb_df['ΔGWST'] = wb_df['GWST'].diff()

    # fill NaN values with 0
    wb_df.fillna(0, inplace=True)

    return wb_df

def calc_wb(df, sim_start=None):
    ''' 
    Calculate water balance components and residuals
    df (DataFrame): DataFrame containing water balance components
    sim_start (float): starting year of the actual simulation after spinup, 
                        leaving as None means all years are included in calcs
    
    returns: DataFrame with water balance components and residuals
    '''
    # make copy of df
    df_copy = df.copy()

    # if sim_start is not None, filter the DataFrame to include only rows after sim_start year
    if sim_start is not None:
        df_copy = df_copy[df_copy['Y'] >= sim_start]

    # Calculate components for each row
    inputs = df_copy['PRCP']
    outputs = df_copy['WYLD'] + df_copy['DPRK'] + df_copy['ET']
    storage = df_copy['ΔSNOF'] + df_copy['ΔSW'] + df_copy['ΔSWLT'] + df_copy['ΔGWST']

        # DIAGNOSTIC OUTPUT
    print(f"Total PRCP: {inputs.sum():.1f}")
    print(f"Total RFV: {df_copy['RFV'].sum():.1f}")
    print(f"Total WYLD: {df_copy['WYLD'].sum():.1f}")
    print(f"Total DPRK: {df_copy['DPRK'].sum():.1f}")
    print(f"Total ET: {df_copy['ET'].sum():.1f}")
    print(f"Total PET: {df_copy['PET'].sum():.1f}")
    print(f"Total ΔStorage: {storage.sum():.1f}")
    print(f"Sum of outputs: {outputs.sum():.1f}")

    # Calculate residual for each row
    df_copy['residual'] = inputs - outputs - storage

    mean_residual = df_copy['residual'].mean()
    total_residual = df_copy['residual'].sum()
    percent_error = total_residual / inputs.sum() * 100

    print(f"mean residual: {mean_residual:.3f}")
    print(f"total residual: {total_residual:.3f}")
    print(f"percent error: {percent_error:.3f}%")

    # Display statistics about the residual
    return percent_error

def component_portion(wb_df, component, sim_start=None):
    '''
    Calculate the portion of each water balance component (outputs and storage) as a percentage of total inputs
    wb_df (DataFrame): DataFrame containing water balance components
    sim_start (float): starting year of the actual simulation after spinup, 
                        leaving as None means all years are included in calcs
    component (str): the component to calculate the portion for ('WYLD', 'DPRK', 'ET', 'storage')
    returns: percent of each component as variables
    '''

    # filter the DataFrame to include only rows after sim_start year
    if sim_start is not None:
        wb_df_copy = wb_df[wb_df['Y'] >= sim_start]

    # Calculate total inputs
    total_inputs = wb_df_copy['PRCP'].sum()

    # calculate component portion
    if component == 'storage' or component == 'Storage':
        # Calculate total storage changes
        total_storage = wb_df_copy['ΔSNOF'].sum() + wb_df_copy['ΔSW'].sum() + wb_df_copy['ΔSWLT'].sum() + wb_df_copy['ΔGWST'].sum()
        portion = total_storage / total_inputs * 100
    else:
        portion = wb_df_copy[component].sum() / total_inputs * 100

    return portion

<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:21: SyntaxWarning: invalid escape sequence '\s'
<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:21: SyntaxWarning: invalid escape sequence '\s'
C:\Users\maya\AppData\Local\Temp\ipykernel_25156\1585931939.py:20: SyntaxWarning: invalid escape sequence '\s'
  sad_df = pd.read_csv(sad_file[0], sep='\s+', skiprows=9, encoding='latin1')
C:\Users\maya\AppData\Local\Temp\ipykernel_25156\1585931939.py:21: SyntaxWarning: invalid escape sequence '\s'
  dhy_df = pd.read_csv(dhy_file[0], sep='\s+', skiprows=8, encoding='latin1')


## Nutrient balance

In [7]:
def nbal_out(fp_run):
    """
    Locate the .OUT file in the given directory, navigate to the last instance of "Total N Balance",
    convert the text to a DataFrame, and compute the residual.
    
    Parameters:
    fp (str): The file path to the directory containing the .OUT file.
    
    Returns:
    pd.DataFrame: A DataFrame containing the computed residual.
    """
    import pandas as pd
    import os
    import re
    import glob
    
    out_files = glob.glob(os.path.join(fp_run, '*.OUT'))
    if not out_files:
        raise FileNotFoundError("No .OUT file found in the specified directory.")
    out_file = out_files[0]  # Take the first match

    with open(out_file, 'r') as file:
        lines = file.readlines()
    
    # Find last instance of "Total N Balance"
    total_n_balance_index = [i for i, line in enumerate(lines) if "TOTAL N BALANCE" in line]
    

    if not total_n_balance_index:
        raise ValueError("No 'Total N Balance' found in the .OUT file.")
    
    last_index = total_n_balance_index[-1]
    

    # Convert text to DataFrame
    data = {}
    for line in lines[last_index + 1:]:
        # Stop parsing after RSDN is found
        if 'RSDN' in line:
            matches = re.findall(r'(\w+)\s*=\s*([-+]?\d*\.?\d+D?[+-]?\d*)', line)
            for var, value in matches:
                value = value.replace('D', 'E')
                data[var] = float(value)
            break  # Stop after RSDN line
        matches = re.findall(r'(\w+)\s*=\s*([-+]?\d*\.?\d+D?[+-]?\d*)', line)
        for var, value in matches:
            value = value.replace('D', 'E')
            data[var] = float(value)
    
    df_n_out = pd.DataFrame([data])

    # filter non-N columns
    cols = ['PER', 'DF', 'BSN', 'PCP', 'DN', 'NFIX', 'VOL', 'FNO', 'FNMN', 'FNMA',
            'BURN', 'DPKN', 'PSON', 'YNWN', 'YN', 'WYLN', 'YLN', 'FSN', 'TLMN',
            'TSMN', 'TRMN', 'TRON', 'RSDN']
    df_n_out = df_n_out[[col for col in cols if col in df_n_out.columns]]

    # compute residual
    out_inputs = df_n_out['PCP'] + df_n_out['BSN'] + df_n_out['NFIX'] + df_n_out['FNO'] + df_n_out['FNMN'] + df_n_out['FNMA']
    out_outputs = df_n_out['DN'] + df_n_out['VOL'] + df_n_out['DPKN'] + df_n_out['YN'] + df_n_out['WYLN'] + df_n_out['YLN'] + df_n_out['FSN']
    residual = out_inputs - out_outputs

    return df_n_out, residual

def pbal_out(fp_run):
    """
    Locate the .OUT file in the given directory, navigate to the last instance of "Total N Balance",
    convert the text to a DataFrame, and compute the residual.
    
    Parameters:
    fp (str): The file path to the directory containing the .OUT file.
    
    Returns:
    pd.DataFrame: A DataFrame containing the computed residual.
    """
    import pandas as pd
    import os
    import re
    import glob
    
    out_files = glob.glob(os.path.join(fp_run, '*.OUT'))
    if not out_files:
        raise FileNotFoundError("No .OUT file found in the specified directory.")
    out_file = out_files[0]  # Take the first match

    with open(out_file, 'r') as file:
        lines = file.readlines()
    
    # Find last instance of "Total N Balance"
    total_p_balance_index = [i for i, line in enumerate(lines) if "P BALANCE (kg)" in line]
    

    if not total_p_balance_index:
        raise ValueError("No 'Total N Balance' found in the .OUT file.")
    
    last_index = total_p_balance_index[-1]
    

    # Convert text to DataFrame
    data = {}
    for line in lines[last_index + 1:]:
        # Stop parsing after FTOT is found
        if 'FTOT' in line:
            matches = re.findall(r'(\w+)\s*=\s*([^\s]*)', line)
            for var, value in matches:
                try:
                    data[var] = float(value.replace('D', 'E'))
                except (ValueError, TypeError):
                    data[var] = np.nan
            break  # Stop after FTOT line
        matches = re.findall(r'(\w+)\s*=\s*([^\s]*)', line)
        for var, value in matches:
            try:
                data[var] = float(value.replace('D', 'E'))
            except (ValueError, TypeError):
                data[var] = np.nan
    
    df_p_out = pd.DataFrame([data])

    # filter non-N columns
    cols = ['PER', 'DF', 'BTOT', 'YI', 'YO', 'QI', 'QO','PRK',
            'YWN', 'YLD', 'FPML', 'FPO', 'SPOU', 'PSOP', 'FTOT']
    df_p_out = df_p_out[[col for col in cols if col in df_p_out.columns]]

    # report columns with NaN values
    if df_p_out.isnull().values.any():
        print("Warning: The following columns contain NaN values:")
        print(df_p_out.columns[df_p_out.isnull().any()].tolist())
    
    # fill NaN values with 0
    # df_p_out.fillna(0, inplace=True)
    
    # compute residual
    out_inputs = df_p_out['BTOT'] + df_p_out['YI'] + df_p_out['QI'] + df_p_out['FPO'] + df_p_out['PSOP'] + df_p_out['FPML']
    out_outputs = df_p_out['YO'] + df_p_out['QO'] + df_p_out['PRK'] + df_p_out['YWN'] + df_p_out['YLD'] + df_p_out['FTOT'] + df_p_out['SPOU']
    residual = out_inputs - out_outputs

    return df_p_out, residual

## Crop Stress

In [8]:
import os
import pandas as pd
import re

def avg_stress(directory):
    '''
    Locate the .OUT file in the given directory, read the last line containing "STRESS",
    and extract the annual averages of BIOM and ROOT stress into a DataFrame.
    
    Parameters:
    directory (str): The file path to the directory containing the .OUT file.
    '''
    # Look for .OUT file in directory
    out_file = None
    for file in os.listdir(directory):
        if file.endswith('.OUT'):
            out_file = file
            break
    
    if not out_file:
        return None
    
    # Open and read the file
    with open(os.path.join(directory, out_file), 'r') as f:
        lines = f.readlines()
    
    # Find last line containing "STRESS"
    last_stress_line = None
    for line in lines:
        if "STRESS" in line:
            last_stress_line = line.strip()
    
    if not last_stress_line:
        return None
    
    # Remove "STRESS" and split by (BIOM) and (ROOT)
    line_without_stress = last_stress_line.replace("STRESS", "").strip()
    
    # Split into BIOM and ROOT sections
    biom_section = re.search(r'\(BIOM\)(.*?)\(ROOT\)', line_without_stress)
    root_section = re.search(r'\(ROOT\)(.*?)$', line_without_stress)
    
    # Extract values and create dataframe
    data = {}
    
    if biom_section:
        biom_data = biom_section.group(1).strip()
        biom_values = re.findall(r'(\w+)=\s*([\d.]+)', biom_data)
        for param, value in biom_values:
            data[f'BIOM_{param}'] = float(value)
    
    if root_section:
        root_data = root_section.group(1).strip().rstrip('D')  # Remove trailing 'D'
        root_values = re.findall(r'(\w+)=\s*([\d.]+)', root_data)
        for param, value in root_values:
            data[f'ROOT_{param}'] = float(value)
    
    return data

## Crop Yield

In [9]:
import os
import pandas as pd

def avg_cropyld(fp, sim_start=None):
    ''' 
    Locate the .ACY file in the given directory, read as a DataFrame, and extract the annual averages of crop yield for both grain, fiber, etc. and forage.

    Parameters:
    fp (str): The file path to the directory containing the .ACY file.
    sim_start (float): Starting year of the actual simulation after spinup, leaving as None means all years are included in calcs.
    '''

    # Look for .ACY file in directory
    acy_file = None
    for file in os.listdir(fp):
        if file.endswith('.ACY'):
            acy_file = file
            break
    if not acy_file:
        return None

    # Open and read the file
    acy_df = pd.read_csv(os.path.join(fp, acy_file), sep='\s+', skiprows=8)
    if sim_start is not None:
        crop_yld_df = acy_df[acy_df['YR'] >= sim_start]
    else:
        crop_yld_df = acy_df

    yld_data = {}

    # Calculate simulation annual average by each crop type (CPNM)
    if not crop_yld_df.empty:
        crop_yld_df = crop_yld_df.groupby(['CPNM']).mean(numeric_only=True).reset_index()

        # Build yld_data: {"crop name": YLDG or YLDF (whichever is not zero)}
        for _, row in crop_yld_df.iterrows():
            crop = row['CPNM']
            yldg = row['YLDG'] if 'YLDG' in row else 0
            yldf = row['YLDF'] if 'YLDF' in row else 0
            # Prefer YLDG if not zero, else YLDF
            if yldg != 0:
                yld_data[crop] = yldg
            elif yldf != 0:
                yld_data[crop] = yldf

    return yld_data

<>:23: SyntaxWarning: invalid escape sequence '\s'
<>:23: SyntaxWarning: invalid escape sequence '\s'
C:\Users\maya\AppData\Local\Temp\ipykernel_25156\3460744301.py:23: SyntaxWarning: invalid escape sequence '\s'
  acy_df = pd.read_csv(os.path.join(fp, acy_file), sep='\s+', skiprows=8)


## Removal

In [ ]:
def generate_loading_table(source_fp):
    '''
    Create a dataframe with loading components from daily summary output files (.DMR)
    source_fp (str): path to the directory containing the .SAD and .DHY files
    returns: dataframe with loading components
    '''
    
    # locate the .SAD file in the source directory
    sad_file = glob.glob(os.path.join(source_fp, '*.SAD'))


    # read in files as df
    sad_df = pd.read_csv(sad_file[0], sep='\s+', skiprows=9, encoding='latin-1')

    # create new df with only the variables of interest
    load_df = sad_df[['#','Y','M','D','PRCP','WYLD','Q','USLE', 'MUSL', 'YP', 'YN', 'QP', 'QN', 'WS', 'NS', 'TS', 'BIOM']]
    load_df = load_df.rename(columns={
        '#':'subarea-id', 'Y':'year', 'M':'month', 'D':'day', 
        'PRCP':'PRCPmm', 'WYLD':'WYLDmm', 'Q':'Qmm', 
        'USLE':'USLEt/ha', 'MUSL':'MUSLEt/ha',
        'YP':'YPkg/ha', 'YN':'YNkg/ha', 'QP':'QPkg/ha', 'QN':'QNkg/ha',
        'WS':'WS', 'NS':'NS', 'TS':'TS', 'BIOM':'BIOM'
    })
    
    load_df['Nkg/ha'] = load_df['YNkg/ha'] + load_df['QNkg/ha']
    load_df['Pkg/ha'] = load_df['YPkg/ha'] + load_df['QPkg/ha']

    # convert Y M D to datetime
    load_df['date'] = pd.to_datetime(load_df[['year', 'month', 'day']])

    return load_df

def calc_removal_efficiency(base_df, bmp_df, component):
    '''
    Calculate the removal efficiency of a BMP for a specific nutrient or sediment
    base_df (DataFrame): df with baseline loading data (from get_loading_table)
    bmp_df (DataFrame): df with bmp loading data (from get_loading_table)
    component (str): the component to calculate the removal efficiency for ('Y', 'YN', 'YP', 'QN', 'QP')

    returns: removal efficiency as a percentage
    '''
    
    # Calculate total loading for baseline and BMP
    base_total = base_df[component].sum()
    bmp_total = bmp_df[component].sum()

    # Calculate removal efficiency
    if base_total == 0:
        return 0.0  # Avoid division by zero

    removal_efficiency = (base_total - bmp_total) / base_total * 100

    return removal_efficiency

def create_removal_efficiency_table(source_fp):
    '''
    Create a table of BMP removal efficiencies relative to baseline
    source_fp (str): path to the directory containing scenario folders
    returns: DataFrame with removal efficiencies for each BMP scenario
    '''
    
    # get all run names in base fp
    runs = [f for f in os.listdir(source_fp) if os.path.isdir(os.path.join(source_fp, f))]
    
    # components to calculate removal efficiency for
    removal_components = ['Qmm', 'QNkg/ha', 'YNkg/ha','Nkg/ha', 'QPkg/ha', 'YPkg/ha', 'Pkg/ha', 'MUSLEt/ha', 'USLEt/ha']
    
    base_run = None
    processed_dfs = {}
    
    # find baseline and process all data
    for run in runs:
        folder_path = os.path.join(source_fp, run)
        df = generate_loading_table(folder_path)
        
        if df.empty:
            print(f"  empty dataframe for {run}")
            continue
        
        # filter out spinup time
        copy_df = df[(df['year'] >= 1990)]
        processed_dfs[run] = copy_df
        
        # identify baseline run
        if 'base' in run.lower():
            base_run = run
    
    if base_run is None:
        print("Warning: No baseline scenario found (looking for 'base' in folder name)")
        return pd.DataFrame()
    
    if base_run not in processed_dfs:
        print(f"Error: Baseline scenario '{base_run}' has empty dataframe")
        return pd.DataFrame()
    
    # calculate removal efficiency for each run against the baseline
    removal_efficiencies = {}
    base_df = processed_dfs[base_run]
    
    for run in processed_dfs:
        if run != base_run:
            removal_efficiencies[run] = {}
            for component in removal_components:
                efficiency = calc_removal_efficiency(base_df, processed_dfs[run], component)
                removal_efficiencies[run][component] = efficiency
    
    # convert to DataFrame
    if removal_efficiencies:
        results_df = pd.DataFrame(removal_efficiencies).T
        results_df.index.name = 'Scenario'
        
        # round to 1 decimal place for cleaner display
        results_df = results_df.round(1)
        
        return results_df
    else:
        print("No BMP scenarios found to compare against baseline")
        return pd.DataFrame()


<>:57: SyntaxWarning: invalid escape sequence '\s'
<>:57: SyntaxWarning: invalid escape sequence '\s'
C:\Users\maya\AppData\Local\Temp\ipykernel_25156\2791756712.py:57: SyntaxWarning: invalid escape sequence '\s'
  sad_df = pd.read_csv(sad_file[0], sep='\s+', skiprows=9, encoding='latin-1')


## Diagnostics

### WB goals

In [ ]:
import sys
from io import StringIO
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

# Define goal ranges for each region (from CBP)
goal_ranges_rc = {
    'a-rc-base-1990-2000': {'WYLD': (45, 60), 'ET': (40, 55), 'PET': (41, 60)},
    'cp-rc-base-1990-2000': {'WYLD': (30, 41), 'ET': (59, 70), 'PET': (71, 94)},
    'p-rc-base-1990-2000': {'WYLD': (30, 38), 'ET': (63, 70), 'PET': (83, 94)},
    'vr-rc-base-1990-2000': {'WYLD': (37, 60), 'ET': (40, 64), 'PET': (42, 87)}
}

goal_ranges_hay = {
    'a-ha-base-1990-2000': {'WYLD': (44, 59), 'ET': (41, 57), 'PET': (41, 60)},
    'cp-ha-base-1990-2000': {'WYLD': (28, 39.5), 'ET': (61,	72), 'PET': (71, 94)},
    'p-ha-base-1990-2000': {'WYLD': (28, 37), 'ET': (64, 72), 'PET': (83, 94)},
    'vr-ha-base-1990-2000': {'WYLD': (35, 59), 'ET': (41, 72), 'PET': (42, 94)}
}

goal_ranges_pas = {
    'a-pa-base-1990-2000': {'WYLD': (44, 59), 'ET': (41, 57), 'PET': (41, 60)},
    'cp-pa-base-1990-2000': {'WYLD': (28, 39.5), 'ET': (61,	72), 'PET': (71, 94)},
    'p-pa-base-1990-2000': {'WYLD': (28, 37), 'ET': (64, 72), 'PET': (83, 94)},
    'vr-pa-base-1990-2000': {'WYLD': (35, 59), 'ET': (41, 72), 'PET': (42, 94)}
}

goal_ranges_for = {
    'a-fr-base-1990-2000': {'WYLD': (42, 59), 'ET': (41, 59), 'PET': (41, 60)},
    'cp-fr-base-1990-2000': {'WYLD': (23, 37), 'ET': (64, 77), 'PET': (71, 94)},
    'p-fr-base-1990-2000': {'WYLD': (23, 33), 'ET': (68, 77), 'PET': (82, 94)},
    'vr-fr-base-1990-2000': {'WYLD': (30, 59), 'ET': (41, 77), 'PET': (41, 94)}
}

def format_goal_range(component, goals):
    if component in goals:
        min_val, max_val = goals[component]
        return f"{min_val}-{max_val}"
    return 'N/A'

def print_baseline_wb_goals(fp):
    '''
    Print WYLD, ET, and PET components with goal ranges for baseline folders only
    '''
    
    # Determine which goal ranges to use based on filepath
    if "row-crops" in fp or "rc" in fp:
        goal_ranges = goal_ranges_rc
    elif "hayland" in fp or "ha" in fp:
        goal_ranges = goal_ranges_hay
    elif "pasture" in fp or "pa" in fp:
        goal_ranges = goal_ranges_pas
    else:
        goal_ranges = goal_ranges_for

    # Suppress all print output from the processing functions
    old_stdout = sys.stdout
    sys.stdout = StringIO()
    
    try:
        for folder in os.listdir(fp):
            # Only process folders with "base" in the name
            if "base" in folder and os.path.isdir(os.path.join(fp, folder)):
                folder_path = os.path.join(fp, folder)
                
                # Temporarily restore stdout for our prints
                sys.stdout = old_stdout
                print(f"\n{folder}:")
                # Suppress again
                sys.stdout = StringIO()
                
                df = generate_wb_table(folder_path)
                wyld_portion = component_portion(df, 'WYLD', sim_start=1990)
                et_portion = component_portion(df, 'ET', sim_start=1990)
                pet_portion = component_portion(df, 'PET', sim_start=1990)
                
                # Get goal ranges for this folder
                goals = goal_ranges.get(folder, {})
                
                # Restore stdout for our results
                sys.stdout = old_stdout
                
                # Print components with goals
                for comp_name, comp_value in [('WYLD', wyld_portion), ('ET', et_portion), ('PET', pet_portion)]:
                    if comp_name in goals:
                        min_val, max_val = goals[comp_name]
                        goal_text = f"{min_val}-{max_val}"
                        within_goal = "[in-range]" if min_val <= comp_value <= max_val else "[out-of-range]"
                        print(f"{comp_name}: {comp_value:.3f} (Goal: {goal_text}) {within_goal}")
                    else:
                        print(f"{comp_name}: {comp_value:.3f} (Goal: N/A)")
                
                # Suppress again for next iteration
                sys.stdout = StringIO()
                
    finally:
        # Always restore stdout
        sys.stdout = old_stdout

### BMP Table

In [12]:
def create_bmp_table(source_fp):
    """
    Create a table of BMP removal efficiencies relative to baseline with additional metrics.
    
    Parameters
    ----------
    source_fp : str
        Path to the directory containing scenario folders.
    
    Returns
    -------
    pandas.DataFrame
        Removal efficiencies and additional metrics for each BMP scenario.
    """
    # local imports in case they weren't added at file top
    import os
    import numpy as np
    import pandas as pd
    import sys
    from io import StringIO

    # get all run names in base fp
    runs = [f for f in os.listdir(source_fp) if os.path.isdir(os.path.join(source_fp, f))]

    # components to calculate removal efficiency for
    removal_components = ['Qmm', 'QNkg/ha', 'QPkg/ha', 'YNkg/ha', 'YPkg/ha', 'Nkg/ha', 'Pkg/ha', 'MUSLEt/ha', 'USLEt/ha']

    base_run = None
    processed_dfs = {}
    all_results = {}

    # find baseline and process all data
    for run in runs:
        folder_path = os.path.join(source_fp, run)
        df = generate_loading_table(folder_path)

        if df.empty:
            print(f"  empty dataframe for {run}")
            continue

        # filter out spinup time
        copy_df = df[(df['year'] >= 1990)]
        processed_dfs[run] = copy_df

        # identify baseline run
        if 'base' in run.lower():
            base_run = run

        # Suppress print output from functions
        old_stdout = sys.stdout
        sys.stdout = StringIO()

        try:
            # WATER BALANCE
            wb_df = generate_wb_table(folder_path)
            percent_error = calc_wb(wb_df, sim_start=1990)
            filtered_wb = wb_df[wb_df['Y'] >= 1990]

            # totals
            total_prcp = filtered_wb['PRCP'].sum()
            total_et = filtered_wb['ET'].sum()
            total_wyld = filtered_wb['WYLD'].sum()

            # components
            et_portion = (total_et / total_prcp * 100) if total_prcp > 0 else 0
            wyld_portion = (total_wyld / total_prcp * 100) if total_prcp > 0 else 0

            # NUTRIENT BALANCE
            # N
            nbal_out_df, nbal_out_residual = nbal_out(folder_path)
            FSN = nbal_out_df['FSN'].iloc[-1]
            n_out_apex = nbal_out_df['DF'].iloc[0] / FSN * 100

            # P
            pbal_out_df, pbal_out_residual = pbal_out(folder_path)
            FSP = pbal_out_df['FTOT'].iloc[-1]
            p_out_apex = pbal_out_df['DF'].iloc[0] / FSP * 100

            # CROP YIELD
            yld_data = avg_cropyld(folder_path, sim_start=1990)
            if yld_data:
                crop, yld = next(iter(yld_data.items()))
            else:
                crop, yld = None, None

            # STRESS — use your existing avg_stress() (unchanged)
            stress = avg_stress(folder_path) or {}
            # prefer ROOT_* if present; fall back to BIOM_*
            n_stress = stress.get('ROOT_N', stress.get('BIOM_N'))
            p_stress = stress.get('ROOT_P', stress.get('BIOM_P'))

        finally:
            # Restore stdout
            sys.stdout = old_stdout

        # Store additional metrics
        all_results[run] = {
            'WB_error': percent_error,
            'NB_error': n_out_apex,
            'PB_error': p_out_apex,
            'Crop': crop,
            'Yield_t_ha': yld,
            'N_stress': n_stress,
            'P_stress': p_stress,
            'ET_mm': total_et,
            'ET_per': et_portion,
            'WYLD_mm': total_wyld,
            'WYLD_per': wyld_portion
            
        }

    if base_run is None:
        print("Warning: No baseline scenario found (looking for 'base' in folder name)")
        return pd.DataFrame()

    if base_run not in processed_dfs:
        print(f"Error: Baseline scenario '{base_run}' has empty dataframe")
        return pd.DataFrame()

    # calculate removal efficiency for each run against the baseline
    removal_efficiencies = {}
    base_df = processed_dfs[base_run]

    # Include all scenarios (including baseline)
    all_scenarios = list(processed_dfs.keys())

    for run in all_scenarios:
        removal_efficiencies[run] = {}
        if run != base_run:
            # Calculate removal efficiency for BMP scenarios
            for component in removal_components:
                efficiency = calc_removal_efficiency(base_df, processed_dfs[run], component)
                removal_efficiencies[run][component] = efficiency
        else:
            # Set baseline removal efficiency to 0 (no removal)
            for component in removal_components:
                removal_efficiencies[run][component] = 0.0

    # convert to DataFrame and combine with additional metrics
    if removal_efficiencies:
        results_df = pd.DataFrame(removal_efficiencies).T

        # Add additional metrics columns for all scenarios
        for run in all_scenarios:
            if run in all_results:
                results_df.loc[run, 'WB error'] = all_results[run]['WB_error']
                results_df.loc[run, 'NB error'] = all_results[run]['NB_error']
                results_df.loc[run, 'PB error'] = all_results[run]['PB_error']
                results_df.loc[run, 'Crop'] = all_results[run]['Crop']
                results_df.loc[run, 'Crop Yield'] = all_results[run]['Yield_t_ha']
                results_df.loc[run, 'N stress'] = all_results[run]['N_stress']
                results_df.loc[run, 'P stress'] = all_results[run]['P_stress']
                results_df.loc[run, 'ET_mm'] = all_results[run]['ET_mm']
                results_df.loc[run, 'ET_per'] = all_results[run]['ET_per']
                results_df.loc[run, 'WYLD_mm'] = all_results[run]['WYLD_mm']
                results_df.loc[run, 'WYLD_per'] = all_results[run]['WYLD_per']

        results_df.index.name = 'Scenario'

        # round numeric columns to 1 decimal place for cleaner display
        numeric_cols = results_df.select_dtypes(include=[np.number]).columns
        results_df[numeric_cols] = results_df[numeric_cols].round(1)

        return results_df

    else:
        print("No BMP scenarios found to compare against baseline")
        return pd.DataFrame()


## Edit

### Single Scenario

In [ ]:
import shutil
import os
import subprocess

def get_scenario_id(filepath):
    '''get the scenario/bmp identifier from the fp'''
    filepath_lower = filepath.lower()
    
    # Check for scenario patterns - order matters!
    scenarios = ['base', 'cc', 'ma', 'nt', 'gb', 'nm']
    for s in scenarios:
        if f'-{s}' in filepath_lower:
            return s
    
    return None


def _modify_sol_file(filepath, changes):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    # Soil albedo (line 2, columns 17-24)
    if 'soil_albedo' in changes and len(lines) >= 2:
        line = lines[1]
        lines[1] = line[:0] + f"{changes['soil_albedo']:8.2f}" + line[8:]

    # Soil group (line 3, columns 25-32)
    if 'soil_grouping' in changes and len(lines) >= 3:
        line = lines[1]
        lines[1] = line[:8] + f"{changes['soil_grouping']:8.2f}" + line[16:]
    
    # Wilting point (line 6)
    if 'wp' in changes and len(lines) >= 6:
        val = f"{changes['wp']:8.2f}"
        lines[5] = val * 5 + " " * 40 + '\n'
    
    # Field capacity (line 7)  
    if 'fc' in changes and len(lines) >= 7:
        val = f"{changes['fc']:8.2f}"
        lines[6] = val * 5 + " " * 40 + '\n'

    # WOC (line 13)
    if 'woc' in changes and len(lines) >= 13:
        val = f"{changes['woc']:8.2f}"
        lines[12] = val * 5 + " " * 40 + '\n'
    
    # SATC (line 22)
    if 'satc' in changes and len(lines) >= 22:
        val = f"{changes['satc']:8.2f}"
        lines[21] = val * 5 + " " * 40 + '\n'
    
    # HCL (line 23)
    if 'hcl' in changes and len(lines) >= 23:
        val = f"{changes['hcl']:8.2f}"
        lines[22] = val * 5 + " " * 40 + '\n'

    # residue
    if 'residue' in changes and len(lines) >= 19:
        line = lines[18]
        lines[18] =  line[:0] + f"{changes['residue']:8.2f}" + line[8:]

    with open(filepath, 'w') as f:
        f.writelines(lines)

def _modify_opc_file(filepath, changes):
    with open(filepath, 'r') as f:
        lines = f.readlines()

    # get scenario identifiers from fp
    scenario_id = get_scenario_id(filepath)

    # get proportions from changes dict
    fert_proportions = changes.get('fert_proportions', {})

    if scenario_id == 'cc':
        lines = [line for line in lines if 'fertilizer (N)' not in line and 'fertilizer (P)' not in line]
    

    for i, line in enumerate(lines):
        # N fertilizer
        if 'fertilizer (N' in line and scenario_id and f'n_fert_{scenario_id}' in changes:
            dose_id = extract_dose_id(line, 'N')
            amt = get_dose_amt(changes[f'n_fert_{scenario_id}'], dose_id, fert_proportions)
            lines[i] = line[:29] + f"{amt:8.2f}" + line[37:]

        # P fertilizer
        if 'fertilizer (P' in line and scenario_id and f'p_fert_{scenario_id}' in changes:
            dose_id = extract_dose_id(line, 'P')
            amt = get_dose_amt(changes[f'p_fert_{scenario_id}'], dose_id, fert_proportions)
            lines[i] = line[:29] + f"{amt:8.2f}" + line[37:]

        # generic changes
        elif 'harvest' in line and 'harvest_efficiency' in changes:
            lines[i] = line[:80] + f"{changes['harvest_efficiency']:5.3f}" + line[85:]
        elif 'manure' in line and 'manure_incorp' in changes:
            lines[i] = line[:29] + f"{changes['manure_incorp']:8.2f}" + line[37:]
        elif 'manure' in line and 'manure_type' in changes:
            lines[i] = line[:9] + f"{changes['manure_type']:8.2f}" + line[17:]
        elif 'plant soybeans' in line and 'phu_soy' in changes:
            lines[i] = line[:29] + f"{changes['phu_soy']:8.2f}" + line[37:]
        elif 'plant corn' in line and 'phu_corn' in changes:
            lines[i] = line[:29] + f"{changes['phu_corn']:8.2f}" + line[37:]        
        elif 'plant wheat' in line and 'phu_wheat' in changes:
            lines[i] = line[:29] + f"{changes['phu_wheat']:8.2f}" + line[37:]
        elif 'plant alfalfa' in line and 'phu_alfalfa' in changes:
            lines[i] = line[:29] + f"{changes['phu_alfalfa']:8.2f}" + line[37:]
        elif 'plant soybeans' in line and 'density_soy' in changes:
            lines[i] = line[:61] + f"{changes['density_soy']:8.2f}" + line[69:]
        elif 'plant wheat' in line and 'density_wheat' in changes:
            lines[i] = line[:61] + f"{changes['density_wheat']:8.2f}" + line[69:]

    # add n_fert & p_fert lines if not already there (cc only)
    if scenario_id == 'cc':
        has_n_fert = any('fertilizer (N' in line for line in lines)
        has_p_fert = any('fertilizer (P' in line for line in lines)
        
        lines_to_add = []
        
        if not has_n_fert and 'n_fert_cc' in changes and changes['n_fert_cc'] != 0:
            lines_to_add.append(f"  1  4 11  580    0    1   52    {changes['n_fert_cc']:5.2f}    0.00    0.00     0.0  0.0000    0.00   0.000       fertilizer (N)\n")
        
        if not has_p_fert and 'p_fert_cc' in changes and changes['p_fert_cc'] != 0:
            lines_to_add.append(f"  1  4 11  580    0    1   52    {changes['p_fert_cc']:5.2f}    0.00    0.00     0.0  0.0000    0.00   0.000       fertilizer (P)\n")
        
        for idx, line in enumerate(lines_to_add):
            lines.insert(4 + idx, line)
    
    with open(filepath, 'w') as f:
        f.writelines(lines)

def extract_dose_id(line, nutrient):
    '''extract dose id from fertilizer line'''
    import re
    pattern = rf'fertilizer \({nutrient}(\d+)\)'
    match = re.search(pattern, line)
    if match:
        return f'{nutrient}{match.group(1)}'
    return nutrient # leave as is if no dose 

def get_dose_amt(total_amt, dose_id, fert_proportions):
    '''get amount for specific dose id based on proportion, full amt if no dose id'''
    # no proportions defined so apply full amount
    if not fert_proportions:
        return total_amt
    
    # proportions set
    proportion = fert_proportions.get(dose_id, 0.0)
    return total_amt * proportion


def _modify_sub_file(filepath, changes):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    scenario_id = get_scenario_id(filepath)

    # PEC is the first value on line 10
    if len(lines) >= 10:
        #change all PEC
        if 'pec' in changes:
            line = lines[9]
            parts = line.split()
            if len(parts) > 0:
                parts[0] = f"{changes['pec']:.2f}"
                lines[9] = "         " + "      ".join(parts) + "\n"
        # BMP-specific PEC
        elif scenario_id and f'pec_{scenario_id}' in changes:
            line = lines[9]  # Line 10 (0-indexed)
            parts = line.split()
            if len(parts) > 0:
                parts[0] = f"{changes[f'pec_{scenario_id}']:.2f}"
                lines[9] = "         " + "      ".join(parts) + "\n"
        elif 'manning_n' in changes:
            line = lines[3]  # Line 4 (0-indexed)
            lines[3] = line[:76] + f"{changes['manning_n']:4.2f}" + line[80:]
    
    with open(filepath, 'w') as f:
        f.writelines(lines)

def _modify_cont_file(filepath, changes):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    scenario_id = get_scenario_id(filepath)

    # PEC is the first value on line 10
    if len(lines) >= 6:
        #change all PEC
        if 'drv' in changes:
            line = lines[5]
            parts = line.split()
            if len(parts) > 0:
                parts[0] = f"{changes['drv']}"
                lines[5] = "       " + "       ".join(parts) + "\n"

    with open(filepath, 'w') as f:
        f.writelines(lines)


### Batch-Modify

In [ ]:
import os
import shutil

def batch_modify_scenarios(scenarios_folder, changes, new_folder_suffix=None):
    """
    Duplicate entire scenarios folder and modify all scenarios within it
    
    Args:
        scenarios_folder: Path to folder containing scenario subfolders
        new_folder_suffix: Suffix to add to the duplicated folder name
        changes: Dict of modifications to apply to all scenarios
    """
    # Create new folder name and duplicate entire folder
    parent_dir = os.path.dirname(scenarios_folder)
    original_folder_name = os.path.basename(scenarios_folder)
    new_folder_name = f"{original_folder_name}_{new_folder_suffix}"
    new_folder_path = os.path.join(parent_dir, new_folder_name)
    
    shutil.copytree(scenarios_folder, new_folder_path)
    
    scenario_count = 0
    
    # Now modify all scenarios within the duplicated folder
    for item in os.listdir(new_folder_path):
        scenario_path = os.path.join(new_folder_path, item)
        
        # Check if it's a directory (scenario folder)
        if os.path.isdir(scenario_path):
            modify_single_scenario_inplace(scenario_path, changes)
            scenario_count += 1
    
    print(f"Modified {scenario_count} scenarios in: {new_folder_path}")

def modify_single_scenario_inplace(scenario_path, changes):
    """Modify a scenario folder in place (no duplication needed)"""
    import subprocess
    
    # DELETE OLD OUTPUT FILES FIRST
    output_extensions = ['.OUT', '.AWP', '.SAD', '.DHY', '.ACY', '.DMR']
    deleted_count = 0
    for file in os.listdir(scenario_path):
        if any(file.upper().endswith(ext) for ext in output_extensions):
            try:
                os.remove(os.path.join(scenario_path, file))
                deleted_count += 1
            except:
                pass
    
    # Process files in this scenario
    for file in os.listdir(scenario_path):
        filepath = os.path.join(scenario_path, file)
        
        if file.lower().endswith('.sol'):
            _modify_sol_file(filepath, changes)
        elif file.lower().endswith('.opc'):
            _modify_opc_file(filepath, changes)
        elif file.lower().endswith('.sub'):
            _modify_sub_file(filepath, changes)
        elif file.lower() == 'apexcont.dat':
            _modify_cont_file(filepath, changes)
    
    
    # Run apex_mks
    original_dir = os.getcwd()
    os.chdir(scenario_path)
    
    result = subprocess.run(['apex-mks'], shell=True, capture_output=True, text=True)


    os.chdir(original_dir)

